# Choice Complexity in LLMs — Project Starter Notebook

This notebook is the **main coding entry point** for the first paper.

It is designed to help us move from project setup to a working research pipeline on **AmbigNQ**.

The notebook is intentionally organized as a professional research workflow:

1. environment setup
2. dataset loading and inspection
3. candidate generation scaffolding
4. embeddings and semantic similarity
5. Choice Complexity Index (CCI)
6. baseline selectors
7. evaluation and result tables
8. saving outputs for later analysis and paper writing

At this stage, **one notebook is sufficient to start professionally**, as long as the code inside it is modular and easy to migrate into `src/` later. As the project matures, the reusable code should move into Python modules and the notebook should become an orchestration and analysis layer.


## Research objective for this notebook

We want to test a simple but publishable question:

> Given an ambiguous question, if an LLM generates a candidate answer set, can a complexity-aware selector produce a better **fixed-size final set** than simple baselines?

Dataset choice: **AmbigNQ**

Why this notebook matters:
- it starts the real codebase,
- it gives us inspectable examples,
- it provides an initial evaluation scaffold,
- and it will later support result generation for the paper.


## 1. Setup

This cell clones the repository and installs the dependencies from `requirements.txt`.


In [ ]:
!git clone https://github.com/soroushbagheri/choice-complexity-llm.git
%cd choice-complexity-llm
!pip install -q -r requirements.txt

import os
import json
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

print('Environment ready.')


## 2. Global configuration

These are the first-paper defaults. We keep them simple and explicit.


In [ ]:
SEED = 42
MAX_INSPECTION_EXAMPLES = 5
DEV_SUBSET_SIZE = 50
CANDIDATE_SET_SIZE = 10
FINAL_SET_SIZE = 4
HF_CONFIG = 'light'
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
RESULTS_DIR = 'results/project_starter'

os.makedirs(RESULTS_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)

CONFIG = {
    'seed': SEED,
    'dataset': 'sewon/ambig_qa',
    'hf_config': HF_CONFIG,
    'candidate_set_size': CANDIDATE_SET_SIZE,
    'final_set_size': FINAL_SET_SIZE,
    'embedding_model': EMBEDDING_MODEL_NAME,
    'results_dir': RESULTS_DIR,
}

CONFIG


## 3. Load AmbigNQ

We start with the Hugging Face version and use the `light` configuration first.

The initial goal is not to solve the full task immediately. The goal is to understand the dataset structure and make sure the pipeline can read it cleanly.


In [ ]:
dataset = load_dataset('sewon/ambig_qa', HF_CONFIG)
dataset


In [ ]:
train_ds = dataset['train']
valid_ds = dataset['validation']

print('Train size:', len(train_ds))
print('Validation size:', len(valid_ds))
print('Columns:', train_ds.column_names)

train_ds[0]


## 4. Inspect a few examples carefully

Before writing the model code, inspect the data like a researcher. We care especially about:

- the original ambiguous question
- the annotation structure
- the disambiguated QA pairs

This step is important because our generation and evaluation logic must align with the dataset semantics.


In [ ]:
def inspect_example(example: Dict[str, Any]) -> None:
    print('=' * 100)
    print('ID:', example.get('id'))
    print('QUESTION:', example.get('question'))
    print()

    annotations = example.get('annotations', [])
    print('N_ANNOTATIONS:', len(annotations))
    if annotations:
        print('FIRST ANNOTATION KEYS:', list(annotations[0].keys()))
    print()

    qa_pairs = example.get('qaPairs', [])
    print('N_QA_PAIRS:', len(qa_pairs))
    for i, qa in enumerate(qa_pairs[:5]):
        print(f'  QA_PAIR {i + 1}')
        print('    disambiguated_question:', qa.get('question'))
        print('    answer:', qa.get('answer'))
        print()

for i in range(MAX_INSPECTION_EXAMPLES):
    inspect_example(train_ds[i])


## 5. Build a clean project view of each example

We now convert raw dataset examples into a structure that is easier to use inside the project.

The project-level fields we want are:
- id
- ambiguous question
- list of disambiguated questions
- list of acceptable answers

This is the form that later generation, selection, and evaluation functions should consume.


In [ ]:
@dataclass
class AmbigExample:
    example_id: str
    question: str
    disambiguated_questions: List[str]
    reference_answers: List[Any]


def parse_ambignq_example(raw: Dict[str, Any]) -> AmbigExample:
    qa_pairs = raw.get('qaPairs', [])
    disamb_qs = []
    ref_answers = []

    for qa in qa_pairs:
        if qa.get('question') is not None:
            disamb_qs.append(qa.get('question'))
        if qa.get('answer') is not None:
            ref_answers.append(qa.get('answer'))

    return AmbigExample(
        example_id=str(raw.get('id')),
        question=str(raw.get('question')),
        disambiguated_questions=disamb_qs,
        reference_answers=ref_answers,
    )

parsed_examples = [parse_ambignq_example(train_ds[i]) for i in range(min(DEV_SUBSET_SIZE, len(train_ds)))]
parsed_examples[0]


## 6. Candidate generation scaffold

This is where the real LLM pipeline will plug in.

For now, the notebook gives a **professional scaffold** rather than pretending the full generation system is complete. That is the right way to build research code: first define the interface, then connect the backend.

You can later connect:
- OpenAI API
- local Hugging Face models
- cached generations

For the first iteration, we keep a mock generator so that the rest of the pipeline can be developed immediately.


In [ ]:
GENERATION_PROMPT_TEMPLATE = '''
You are given an ambiguous open-domain question.
Generate {k} distinct plausible short answer candidates.
Avoid duplicates and near-duplicates.
Return one candidate per line.

Question: {question}
'''

def build_generation_prompt(question: str, k: int = CANDIDATE_SET_SIZE) -> str:
    return GENERATION_PROMPT_TEMPLATE.format(question=question, k=k)

def mock_generate_candidates(example: AmbigExample, k: int = CANDIDATE_SET_SIZE) -> List[Dict[str, Any]]:
    refs = example.reference_answers
    flattened = []

    for ans_group in refs:
        if isinstance(ans_group, list):
            flattened.extend([str(x) for x in ans_group])
        else:
            flattened.append(str(ans_group))

    flattened = [x.strip() for x in flattened if str(x).strip()]
    flattened = list(dict.fromkeys(flattened))

    candidates = []
    for i in range(k):
        if i < len(flattened):
            text = flattened[i]
            score = 1.0 - i * 0.03
        else:
            text = f'{example.question} :: candidate_{i}'
            score = max(0.1, 0.5 - i * 0.02)

        candidates.append({
            'candidate_id': i,
            'text': text,
            'score': float(score),
        })
    return candidates

demo_prompt = build_generation_prompt(parsed_examples[0].question, CANDIDATE_SET_SIZE)
print(demo_prompt)
print()
mock_generate_candidates(parsed_examples[0], CANDIDATE_SET_SIZE)[:5]


## 7. Embeddings and semantic similarity

We need semantic similarity to measure redundancy between candidate answers.

For now, one sentence-transformer model is enough for the first paper.


In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

def embed_texts(texts: List[str]) -> np.ndarray:
    return np.asarray(embedding_model.encode(texts, normalize_embeddings=True))

sample_candidates = mock_generate_candidates(parsed_examples[0], CANDIDATE_SET_SIZE)
sample_texts = [c['text'] for c in sample_candidates]
sample_embeddings = embed_texts(sample_texts)
sample_embeddings.shape


## 8. Choice Complexity Index (CCI)

Current working formulation:

CCI(S) = w1 * size + w2 * entropy + w3 * redundancy + w4 * ambiguity

We compute every component separately and log them all.


In [ ]:
CCI_WEIGHTS = {
    'size': 0.15,
    'entropy': 0.35,
    'redundancy': 0.30,
    'ambiguity': 0.20,
}

def softmax(x: np.ndarray) -> np.ndarray:
    z = np.exp(x - np.max(x))
    return z / z.sum()

def normalized_entropy(scores: List[float]) -> float:
    probs = softmax(np.asarray(scores, dtype=float))
    h = -np.sum(probs * np.log(probs + 1e-12))
    return float(h / np.log(len(scores))) if len(scores) > 1 else 0.0

def redundancy_score(embeddings: np.ndarray) -> float:
    sims = cosine_similarity(embeddings)
    vals = []
    n = sims.shape[0]
    for i in range(n):
        for j in range(i + 1, n):
            vals.append((sims[i, j] + 1.0) / 2.0)
    return float(np.mean(vals)) if vals else 0.0

def top_option_ambiguity(scores: List[float]) -> float:
    ordered = sorted(scores, reverse=True)
    if len(ordered) < 2:
        return 0.0
    margin = ordered[0] - ordered[1]
    return float(1 / (1 + np.exp(3 * margin)))

def compute_cci(candidates: List[Dict[str, Any]], total_candidate_size: int = CANDIDATE_SET_SIZE) -> Dict[str, float]:
    texts = [c['text'] for c in candidates]
    scores = [float(c['score']) for c in candidates]
    embeddings = embed_texts(texts)

    size_term = len(candidates) / total_candidate_size
    entropy_term = normalized_entropy(scores)
    redundancy_term = redundancy_score(embeddings)
    ambiguity_term = top_option_ambiguity(scores)

    cci = (
        CCI_WEIGHTS['size'] * size_term
        + CCI_WEIGHTS['entropy'] * entropy_term
        + CCI_WEIGHTS['redundancy'] * redundancy_term
        + CCI_WEIGHTS['ambiguity'] * ambiguity_term
    )

    return {
        'cci': float(cci),
        'size_term': float(size_term),
        'entropy_term': float(entropy_term),
        'redundancy_term': float(redundancy_term),
        'ambiguity_term': float(ambiguity_term),
    }

compute_cci(sample_candidates)


## 9. Baseline selectors

All selectors must return the same final number of options. This is one of the most important design choices in the project.


In [ ]:
def select_topk(candidates: List[Dict[str, Any]], k: int = FINAL_SET_SIZE) -> List[Dict[str, Any]]:
    return sorted(candidates, key=lambda x: x['score'], reverse=True)[:k]

def select_random(candidates: List[Dict[str, Any]], k: int = FINAL_SET_SIZE) -> List[Dict[str, Any]]:
    return random.sample(candidates, k)

def select_diverse(candidates: List[Dict[str, Any]], k: int = FINAL_SET_SIZE) -> List[Dict[str, Any]]:
    remaining = candidates[:]
    selected = [max(remaining, key=lambda x: x['score'])]
    remaining.remove(selected[0])

    while len(selected) < k and remaining:
        selected_texts = [x['text'] for x in selected]
        selected_embs = embed_texts(selected_texts)

        best_candidate = None
        best_value = None
        for cand in remaining:
            cand_emb = embed_texts([cand['text']])
            sims = cosine_similarity(cand_emb, selected_embs)[0]
            value = np.max(sims)
            if best_value is None or value < best_value:
                best_value = value
                best_candidate = cand

        selected.append(best_candidate)
        remaining.remove(best_candidate)

    return selected

def select_cci_aware(candidates: List[Dict[str, Any]], k: int = FINAL_SET_SIZE, penalty_strength: float = 0.55) -> List[Dict[str, Any]]:
    ranked = sorted(candidates, key=lambda x: x['score'], reverse=True)
    selected = []

    while len(selected) < k and ranked:
        best_item = None
        best_value = -1e9

        for cand in ranked:
            if not selected:
                value = cand['score']
            else:
                sel_embs = embed_texts([x['text'] for x in selected])
                cand_emb = embed_texts([cand['text']])
                sims = cosine_similarity(cand_emb, sel_embs)[0]
                redundancy_penalty = float(np.mean((sims + 1.0) / 2.0))
                value = cand['score'] - penalty_strength * redundancy_penalty

            if value > best_value:
                best_value = value
                best_item = cand

        selected.append(best_item)
        ranked.remove(best_item)

    return selected

selector_demo = {
    'topk': select_topk(sample_candidates),
    'random': select_random(sample_candidates),
    'diverse': select_diverse(sample_candidates),
    'cci_aware': select_cci_aware(sample_candidates),
}
{k: [x['text'] for x in v] for k, v in selector_demo.items()}


## 10. Basic evaluation scaffold

For the starter notebook, we compute:

- full-set CCI
- selected-set CCI
- complexity reduction
- average candidate score
- a simple reference coverage indicator

This is not the final evaluation yet, but it is the right coding scaffold to begin generating results.


In [ ]:
def flatten_reference_answers(ref_answers: List[Any]) -> List[str]:
    out = []
    for item in ref_answers:
        if isinstance(item, list):
            out.extend([str(x).strip().lower() for x in item if str(x).strip()])
        else:
            val = str(item).strip().lower()
            if val:
                out.append(val)
    return list(dict.fromkeys(out))

def reference_coverage(selected: List[Dict[str, Any]], example: AmbigExample) -> int:
    refs = set(flatten_reference_answers(example.reference_answers))
    sel = set([x['text'].strip().lower() for x in selected])
    return int(len(refs.intersection(sel)) > 0)

def evaluate_one_example(example: AmbigExample) -> List[Dict[str, Any]]:
    candidates = mock_generate_candidates(example, CANDIDATE_SET_SIZE)
    full_stats = compute_cci(candidates)

    strategies = {
        'topk': select_topk,
        'random': select_random,
        'diverse': select_diverse,
        'cci_aware': select_cci_aware,
    }

    rows = []
    for name, fn in strategies.items():
        selected = fn(candidates, FINAL_SET_SIZE)
        sel_stats = compute_cci(selected, total_candidate_size=CANDIDATE_SET_SIZE)

        rows.append({
            'example_id': example.example_id,
            'strategy': name,
            'question': example.question,
            'full_cci': full_stats['cci'],
            'selected_cci': sel_stats['cci'],
            'cci_reduction': full_stats['cci'] - sel_stats['cci'],
            'selected_avg_score': float(np.mean([x['score'] for x in selected])),
            'coverage': reference_coverage(selected, example),
            'selected_texts': [x['text'] for x in selected],
        })

    return rows

starter_rows = []
for ex in parsed_examples[:20]:
    starter_rows.extend(evaluate_one_example(ex))

starter_df = pd.DataFrame(starter_rows)
starter_df.head()


## 11. First aggregate results

These are only starter-pipeline results, but they already let us test whether the coding workflow is behaving sensibly.


In [ ]:
summary = starter_df.groupby('strategy')[['selected_cci', 'cci_reduction', 'selected_avg_score', 'coverage']].mean().sort_values('selected_cci')
summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

summary['selected_cci'].plot(kind='bar', ax=axes[0], title='Final CCI')
axes[0].set_ylabel('CCI')
axes[0].tick_params(axis='x', rotation=30)

summary['cci_reduction'].plot(kind='bar', ax=axes[1], title='CCI Reduction')
axes[1].set_ylabel('Reduction')
axes[1].tick_params(axis='x', rotation=30)

summary['coverage'].plot(kind='bar', ax=axes[2], title='Reference Coverage')
axes[2].set_ylabel('Coverage')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## 12. Save outputs

Even at the beginning of a research project, save intermediate results in a clean way. This makes later debugging, analysis, and paper writing much easier.


In [ ]:
summary_path = os.path.join(RESULTS_DIR, 'starter_summary.csv')
rows_path = os.path.join(RESULTS_DIR, 'starter_rows.csv')
config_path = os.path.join(RESULTS_DIR, 'starter_config.json')

summary.to_csv(summary_path)
starter_df.to_csv(rows_path, index=False)
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2)

print('Saved to:')
print(summary_path)
print(rows_path)
print(config_path)


## 13. What is still missing before the real experiment?

This notebook now gives us a strong professional starting point, but the **real paper pipeline** still needs the following coding steps:

1. replace the mock generator with a real LLM generation backend
2. store raw prompts and raw generations
3. normalize answer strings more carefully for evaluation
4. improve utility scoring using model scores or calibrated ranking signals
5. implement ablations over each CCI term
6. build a clean human-evaluation export format
7. move reusable code from this notebook into `src/` modules

That is exactly how a research codebase should evolve: notebook first for speed and clarity, modules second for stability and reproducibility.


## 14. Immediate next coding step

The next real implementation step should be:

> replace `mock_generate_candidates()` with a real generator that queries the target LLM and saves the candidate set for each AmbigNQ example.

Once that works, this notebook becomes a true end-to-end research pipeline.
